# Session 1 · Kernels and NAIF IDs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lasp/spice-tutorials/blob/main/notebooks/01_kernels_and_ids.ipynb)
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/lasp/spice-tutorials/main?labpath=notebooks/01_kernels_and_ids.ipynb)

**SPICE Book Club, homework for the two weeks after Session 1.** About 45 minutes.

By the end you will have:

1. a working SpiceyPy install;
2. the three generic kernels every later notebook uses (LSK, PCK, planetary SPK);
3. a meta-kernel you wrote yourself, and a feel for SPICE's global kernel pool;
4. a working model of NAIF integer IDs;
5. your first geometry query: the Earth–Moon distance over a month;
6. the habit of reading a kernel's comments and coverage before trusting it.

Cells marked **🔮 Predict** ask you to write down a guess *before* running them. The guess is the point; being wrong is how the model gets fixed.

## 0. Setup

Run this first. It works in Colab, Binder and a local clone of the repo: it finds the course helper package `spiceclub` (installing it from GitHub if needed, which also installs SpiceyPy).

In [ ]:
import importlib, importlib.util, subprocess, sys
from pathlib import Path

for p in (Path.cwd(), *Path.cwd().parents):          # local clone or Binder: use the repo's copy
    if (p / "spiceclub" / "kernels.py").is_file():
        sys.path.insert(0, str(p))
        break
if importlib.util.find_spec("spiceclub") is None:     # Colab or a bare environment
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "spiceclub @ git+https://github.com/lasp/spice-tutorials"], check=True)
    importlib.invalidate_caches()

import numpy as np
import matplotlib.pyplot as plt
import spiceypy as sp
from spiceclub import kernels

print("kernel cache:", kernels.cache_dir())

## 1. Install check

SpiceyPy is a thin Python wrapper around NAIF's C library, CSPICE. The wrapper and the library have separate versions; the one that matters for behavior is the CSPICE toolkit version.

In [ ]:
print("SpiceyPy:", sp.__version__)
print("CSPICE:  ", sp.tkvrsn("TOOLKIT"))

## 2. Fetch the generic kernels

Three files from NAIF's `generic_kernels` area cover what nearly every SPICE program needs:

| File | Type | What it holds |
|---|---|---|
| `naif0012.tls` | LSK (text) | the leap-second table, to convert UTC ↔ ET |
| `pck00011.tpc` | PCK (text) | body radii and IAU rotation models |
| `de440s.bsp` | SPK (binary) | positions of the Sun, Moon and planet barycenters, 1849–2150 (~32 MB) |

`kernels.generic()` downloads each one once and caches it.

In [ ]:
paths = kernels.generic()
for kind, path in paths.items():
    print(f"{kind:4s} {path.stat().st_size / 1e6:8.3f} MB  {path}")

**🔮 Predict.** Two of these are text and one is binary. Which do you think is binary, and why would NAIF use different formats for different kernels? Open `naif0012.tls` in a text editor (or run the next cell) to check part of your answer.

In [ ]:
print(paths["lsk"].read_text()[:1500])

## 3. A meta-kernel, written by hand

Loading kernels one by one with `furnsh` works, but real programs list them in a **meta-kernel** (`.tm`): a small text kernel whose only job is to name other kernels. It is the single place to record *which versions* of *which files* an analysis used.

Two details catch everyone once:

- Relative paths in a meta-kernel are resolved against the **current working directory**, not the meta-kernel's own folder. Absolute paths (or `PATH_SYMBOLS`) avoid that.
- Each quoted string is limited to 255 characters, which long absolute paths can exceed. `PATH_SYMBOLS` keeps them short.

In [ ]:
kdir = kernels.cache_dir().resolve().as_posix()

mk_text = f"""KPL/MK

My first meta-kernel (SPICE Book Club, Session 1).

\\begindata

   PATH_VALUES  = ( '{kdir}' )
   PATH_SYMBOLS = ( 'KERNELS' )

   KERNELS_TO_LOAD = ( '$KERNELS/naif0012.tls'
                       '$KERNELS/pck00011.tpc'
                       '$KERNELS/de440s.bsp' )

\\begintext
"""
mk = Path("session01_mine.tm")
mk.write_text(mk_text)
print(mk_text)

**🔮 Predict.** After `furnsh` on this meta-kernel, how many files will SPICE report as loaded with `ktotal("ALL")`?

In [ ]:
sp.kclear()                      # start from an empty kernel pool
sp.furnsh(str(mk))

n = sp.ktotal("ALL")
print("loaded:", n)
for i in range(n):
    file, ftype, source, handle = sp.kdata(i, "ALL")
    print(f"  {ftype:5s} {Path(file).name:20s} loaded via: {Path(source).name if source else '(direct)'}")

The meta-kernel counts itself: four files, not three. `kdata` also reports *which* meta-kernel pulled each file in, which helps when debugging a program that loads several.

### The kernel pool is global state

Everything loaded goes into one process-wide pool. Any function anywhere in your program sees it, and `kclear` empties it for everyone. Try it:

In [ ]:
def some_library_function():
    sp.kclear()                  # a "tidy" helper that cleans up after itself...

print("before:", sp.ktotal("ALL"))
some_library_function()
print("after: ", sp.ktotal("ALL"))
sp.furnsh(str(mk))               # reload for the rest of the notebook

Keep this in mind for services and pipelines: kernel loading belongs in one place at startup, and CSPICE is not thread-safe. We will come back to it in the discussion.

Later kernels win: if two loaded kernels provide the same quantity for the same time (two versions of an SPK, two values of a constant), the one loaded **last** takes precedence. That is why order in `KERNELS_TO_LOAD` matters.

## 4. Names and NAIF IDs

Internally SPICE identifies every body, spacecraft, instrument and site by an integer. Names are a convenience layer on top.

**🔮 Predict** the integer for each of these before running the cell: `SOLAR SYSTEM BARYCENTER`, `SUN`, `EARTH`, `MOON`, `MARS BARYCENTER`, `MARS`, `PHOBOS`, `MAVEN`.

In [ ]:
for name in ["SOLAR SYSTEM BARYCENTER", "SUN", "EARTH BARYCENTER", "EARTH", "MOON",
             "MARS BARYCENTER", "MARS", "PHOBOS", "MAVEN"]:
    print(f"{name:25s} {sp.bodn2c(name):6d}")

The pattern:

| Range | Meaning | Examples |
|---|---|---|
| 0 | Solar system barycenter | |
| 1–9 | Planet-system barycenters | 3 Earth–Moon, 4 Mars |
| 10 | Sun | |
| *P*99 | Planet *P* itself | 399 Earth, 499 Mars |
| *P*01–*P*98 | Natural satellites of planet *P* | 301 Moon, 401 Phobos |
| negative | Spacecraft | −202 MAVEN |
| −*SSS*000 to −*SSS*999 | Instruments and structures on spacecraft −*SSS* | −202xxx on MAVEN |
| 399000+ | Ground stations | DSN stations |

Why both 4 and 499? Because `de440s.bsp` gives the **Mars barycenter** (4) relative to the Sun. The planet's offset from its system barycenter comes from a separate, mission-specific satellite ephemeris. You will hit this in Session 3.

The mapping works both ways, and some lookups fail on purpose:

In [ ]:
print(sp.bodc2n(-202))
print(sp.bodc2n(499))

try:
    sp.bodn2c("MAVEN_IUVS")
except sp.exceptions.NotFoundError as err:
    print("Not found:", "MAVEN_IUVS is not built in; it is defined in the MAVEN frames kernel (exercise 7).")

## 5. First geometry query: Earth–Moon distance

`spkpos(target, et, frame, abcorr, observer)` returns the position of `target` relative to `observer` in `frame` at ephemeris time `et`. To get there from a calendar date we need the LSK (`str2et`), and for the positions, the SPK.

**🔮 Predict.** The mean Earth–Moon distance is about 384 400 km. Over one month, how far from that do you expect the distance to swing: ±1 000 km, ±20 000 km, or ±100 000 km?

In [ ]:
et0 = sp.str2et("2026-11-01T00:00:00")
et1 = sp.str2et("2026-12-01T00:00:00")
ets = np.linspace(et0, et1, 24 * 30 + 1)          # hourly

pos, lt = sp.spkpos("MOON", ets, "J2000", "NONE", "EARTH")
dist = np.linalg.norm(pos, axis=1)
days = (ets - et0) / 86400

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(days, dist / 1e3)
ax.axhline(384.4, color="grey", ls=":", lw=1)
ax.set_xlabel("Days since 2026-11-01 UTC")
ax.set_ylabel("Earth–Moon distance (1000 km)")
ax.set_title("Moon relative to Earth, geometric (no aberration correction)")
ax.grid(alpha=0.3)
plt.show()

i_min, i_max = dist.argmin(), dist.argmax()
print("perigee:", sp.et2utc(ets[i_min], "C", 0), f"{dist[i_min]:,.0f} km")
print("apogee: ", sp.et2utc(ets[i_max], "C", 0), f"{dist[i_max]:,.0f} km")

Notice what went in: two **name strings** (converted to IDs using the built-in table), a **frame** name, a **time** (converted from UTC using the LSK), and an **aberration correction** flag, which we leave at `"NONE"` until Session 3. Notice what did *not* go in: any file name. The query is answered by whatever is in the kernel pool.

`lt` holds the one-way light time in seconds. Take a look at it: what does it tell you about `"NONE"` vs. what an observer on Earth actually sees?

## 6. Read before you trust: comments and coverage

Every kernel carries comments: who made it, from what data, valid for when, with which caveats. In text kernels they sit outside the `\begindata` blocks. In binary kernels they live in a reserved comment area, read with `dafec`.

In [ ]:
def binary_comments(path):
    """Return the comment area of a binary (DAF) kernel such as an SPK or CK."""
    handle = sp.dafopr(str(path))
    lines = []
    try:
        while True:
            n, buf, done = sp.dafec(handle, 100, 1000)
            lines.extend(buf[:n])
            if done:
                break
    finally:
        sp.dafcls(handle)
    return lines

comments = binary_comments(paths["spk"])
print(f"{len(comments)} comment lines; the first 40:\n")
print("\n".join(comments[:40]))

Comments are the kernel's own documentation. Coverage is what the data actually contain. `spkobj` lists every object in an SPK, and `spkcov` gives the time window for one of them.

In [ ]:
ids = sp.spkobj(str(paths["spk"]))
print(f"{len(ids)} objects in {paths['spk'].name}:\n")
for code_ in sorted(ids):
    cover = sp.spkcov(str(paths["spk"]), code_)
    windows = [sp.wnfetd(cover, i) for i in range(sp.wncard(cover))]
    span = ", ".join(f"{sp.et2utc(a, 'C', 0)[:11]} → {sp.et2utc(b, 'C', 0)[:11]}" for a, b in windows)
    try:
        name = sp.bodc2n(code_)
    except sp.exceptions.NotFoundError:
        name = "?"
    print(f"  {code_:5d} {name:25s} {span}")

**🔮 Predict.** Is `MARS` (499) in that list? Is `MAVEN` (−202)? What will happen if you ask for the Moon's position in the year 1800? Run the cell to find out.

In [ ]:
try:
    sp.spkpos("MOON", sp.str2et("1800-01-01"), "J2000", "NONE", "EARTH")
except sp.exceptions.SpiceyError as err:
    print(type(err).__name__)
    print(str(err)[:600])

Read the whole error: SPICE says which body, which time and how many SPK files it searched. That message format is worth getting comfortable with; Session 6 is partly a tour of the common ones.

## 7. Exercises

**7a. MAVEN's instruments.** The MAVEN frames kernel (FK) defines names and IDs for the spacecraft structures and instruments. Load the newest one and list every name–ID pair it adds.

Hints:

- `fk = kernels.maven("fk")` fetches it.
- A text kernel adds variables to the kernel pool. Name–ID pairs live in the pool variables `NAIF_BODY_NAME` and `NAIF_BODY_CODE`; read them with `sp.gcpool` and `sp.gipool`.
- Instrument IDs on MAVEN fall in −202000 to −202999. Do all the entries?

In [ ]:
# your code here

<details>
<summary><b>Solution 7a</b> (try first!)</summary>

```python
fk = kernels.maven("fk")
sp.furnsh(str(fk))

names = sp.gcpool("NAIF_BODY_NAME", 0, 1000)
codes = sp.gipool("NAIF_BODY_CODE", 0, 1000)
for code_, name in sorted(zip(codes, names), reverse=True):
    print(f"{code_:8d}  {name}")

print(sp.bodn2c("MAVEN_IUVS"))   # now this works
```

`gnpool("FRAME_*_NAME", 0, 1000)` lists the *frame* names the FK defines, which is a different (overlapping) list. We will untangle bodies vs. frames in Sessions 4 and 5.
</details>

**7b. Comments of a text kernel.** Print the comment section of the MAVEN FK (everything before the first `\begindata`). Who produced it, and what is the most recent change in its version history?

**7c. Meta-kernel hygiene.** Write `session01_mine.tm` again with the MAVEN FK added. Then edit it to load `de440s.bsp` *twice*, and check `ktotal`. What does SPICE do with the duplicate?

**7d. For discussion.** Pick a SPICE call in code you work on (or one you've seen). Which kernels does it need, which team produces each of them, and how does your code know which versions it loaded?

In [ ]:
# your code here

<details>
<summary><b>Solution 7b and 7c</b></summary>

```python
text = Path(fk).read_text()
print(text.split("\\begindata")[0])
```

```python
sp.kclear()
sp.furnsh(str(paths["spk"]))
sp.furnsh(str(paths["spk"]))
print(sp.ktotal("SPK"))   # 2: SPICE records every furnsh call, duplicates included
sp.unload(str(paths["spk"]))
print(sp.ktotal("SPK"))   # 1: each unload removes one entry
```

Duplicates do no harm to results, but they show that the loaded-kernel list is a record of *calls*, not a set of files. A duplicated or stale entry in a meta-kernel is easy to miss, which is one more reason to print `kdata` at startup in production code.
```
</details>

---

Next session: **Time**. Read NAIF tutorial 04 slides 4–13 (time concepts), then 15 (time) and 16 (LSK and SCLK).